# Тематическое моделирование новостей Lenta.ru с BERTopic

В этой работе я последовательно подбираю конфигурацию `BERTopic` для корпуса `lenta-ru-news`, затем заново обучаю лучший пайплайн прямо в ноутбуке и отдельно разбираю, насколько получившееся разбиение на темы выглядит содержательно.

Эксперименты для выбора конфигурации я запускал отдельным воспроизводимым скриптом `scripts/hw_4/run_experiment.py`. В ноутбук вынесены его результаты, а финальная модель ниже собирается заново.

Сразу зафиксирую метрики, по которым дальше сравниваются конфигурации.

**Topic Diversity**

Пусть есть `K` тем, и для каждой темы взят список топ-слов `W_k`. Тогда

$$
Topic\ Diversity = \frac{\text{число уникальных слов во всех } W_k}{\text{общее число слов во всех } W_k}
$$

Если во всех темах используются разные слова, значение стремится к `1`. Если одни и те же слова постоянно повторяются в нескольких темах, метрика снижается.

В коде ниже для каждой темы я беру `10` топ-слов, поэтому эту же формулу можно читать и так:

$$
Topic\ Diversity = \frac{\left|\bigcup_{k=1}^{K} W_k\right|}{10K}
$$

**UMass Coherence**

Для каждой темы я смотрю на пары топ-слов и проверяю, насколько часто они встречаются в одних и тех же документах:

$$
C_{UMass} = \text{average}\left[\log \frac{D(w_i, w_j) + 1}{D(w_j)}\right]
$$

Здесь `D(w_i, w_j)` — число документов, где пара слов встретилась вместе, а `D(w_j)` — число документов, где встретилось слово `w_j`. Значение обычно отрицательное. Чем оно выше, тем лучше слова внутри темы согласуются друг с другом на уровне корпуса.

**Coverage**

$$
Coverage = \frac{\text{число документов с темой}}{\text{общее число документов}}
$$

В терминах `BERTopic` это доля документов, которые не получили метку `-1`.

**Selection score**

Для итогового ранжирования конфигураций я беру три основные метрики, привожу их к шкале от `0` до `1` внутри набора сравниваемых кандидатов и считаю взвешенную сумму:

$$
m_{norm}(x) = \frac{x - \min(x)}{\max(x) - \min(x)}
$$

$$
Score = 0.5 \cdot Coh_{norm} + 0.3 \cdot Div_{norm} + 0.2 \cdot Cov_{norm}
$$

Максимальный вес у coherence, потому что для тематического моделирования меня прежде всего интересует связность темы. Diversity и coverage тоже важны, но играют вспомогательную роль.
Кроме метрик, я также определил критерии, невыполнение которых сразу делает конфигурацию неподходящей: покрытие не ниже 0.65 и число тем в диапазоне 12..40.

## 1. Загрузка набора данных через Corus

Из корпуса мне нужны `title`, `text` и `topic`. Для экспериментов я использовал подвыборку из `12 000` документов. Этого объёма хватает, чтобы увидеть устойчивые различия между конфигурациями, и при этом ноутбук выполняется от начала до конца на локальной машине.

In [1]:
from __future__ import annotations

import json
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "outputs").exists() else CWD.parent
DATA_PATH = ROOT / "data" / "lenta-ru-news.csv.gz"
RESULTS_PATH = ROOT / "outputs" / "bertopic_search" / "results.json"
SAMPLE_CACHE_PATH = ROOT / "tmp" / "bertopic_search" / "sample_n12000_seed42.pkl"
SAMPLE_SIZE = 12_000
SEED = 42
DATA_URL = "https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz"


def load_sample(sample_size: int = SAMPLE_SIZE, seed: int = SEED) -> pd.DataFrame:
    if SAMPLE_CACHE_PATH.exists():
        return pd.read_pickle(SAMPLE_CACHE_PATH).reset_index(drop=True)

    if not DATA_PATH.exists():
        DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(DATA_URL, DATA_PATH)

    from corus import load_lenta

    rows = []
    for record in load_lenta(DATA_PATH):
        title = (record.title or "").strip()
        text = (record.text or "").strip()
        topic = (record.topic or "").strip()
        if text and topic:
            rows.append({"title": title, "text": text, "topic": topic})

    df = (
        pd.DataFrame(rows)
        .sample(n=sample_size, random_state=seed)
        .reset_index(drop=True)
    )
    SAMPLE_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_pickle(SAMPLE_CACHE_PATH)
    return df


results = json.loads(RESULTS_PATH.read_text())
sample_df = load_sample()
sample_df.head(3)



,title,text,topic,title_only_light,text_only_light,title_text_light,title_text,text_only_lemma,title_only_lemma,title_text_lemma
0,Организатор теракта у итальянского колледжа пр...,"Джованни Вантаджато (Giovanni Vantaggiato), ус...",Мир,организатор теракта у итальянского колледжа пр...,"джованни вантаджато (giovanni vantaggiato), ус...",организатор теракта у итальянского колледжа пр...,Организатор теракта у итальянского колледжа пр...,"джованни вантаджать ( giovanni vantaggiato ) ,...",организатор теракт у итальянский колледж приго...,организатор теракт у итальянский колледж приго...
1,Президент пообещал программу материнского капи...,Президент России Владимир Путин в ходе ежегодн...,Россия,президент пообещал программу материнского капи...,президент россии владимир путин в ходе ежегодн...,президент пообещал программу материнского капи...,Президент пообещал программу материнского капи...,президент россия владимир путин в ход ежегодны...,президент пообещать программа материнский капи...,президент пообещать программа материнский капи...
2,Центр международной торговли построил новое оф...,"ОАО ""Центр международной торговли"" (ЦМТ) ввело...",Дом,центр международной торговли построил новое оф...,"оао ""центр международной торговли"" (цмт) ввело...",центр международной торговли построил новое оф...,Центр международной торговли построил новое оф...,"оао "" центр международный торговля "" ( цмт ) в...",центр международный торговля построить новый о...,центр международный торговля построить новый о...


In [2]:
sample_overview = pd.DataFrame(
    {
        "documents": [len(sample_df)],
        "topics_in_lenta": [sample_df["topic"].nunique()],
        "median_title_len": [sample_df["title"].str.split().map(len).median()],
        "median_text_len": [sample_df["text"].str.split().map(len).median()],
        "p95_text_len": [sample_df["text"].str.split().map(len).quantile(0.95)],
    }
)

topic_distribution = (
    sample_df["topic"]
    .value_counts()
    .rename_axis("topic")
    .reset_index(name="documents")
    .assign(share=lambda x: x["documents"] / len(sample_df))
)

display(sample_overview)
display(topic_distribution.head(10))



,documents,topics_in_lenta,median_title_len,median_text_len,p95_text_len
0,12000,19,7.0,171.0,306.05


,topic,documents,share
0,Россия,2605,0.217083
1,Мир,2225,0.185417
2,Экономика,1293,0.107750
3,Спорт,1047,0.087250
4,Культура,886,0.073833
5,Наука и техника,869,0.072417
6,Бывший СССР,854,0.071167
7,Интернет и СМИ,746,0.062167
8,Из жизни,433,0.036083
9,Силовые структуры,357,0.029750


В получившийся подвыборке 19 уникальных тем из изначального датасета. Также видим, что сохраняется большая разница между количеством документов в каждой из тем.

## 2. Предобработка

В качестве предобработки текста я рассматривал легкую нормализацию и добавление лемматизации как альтернативу.

Предобработка включала нормализацию текста и очистку стоп-слов. Дальше я сравниваю несколько вариантов подготовки текста и оставляю тот, который показал лучше метрики.

Кроме того, здесь мы сравниваем два варианта подачи на вход текста - с заголовком и без него.



In [3]:
import re

import nltk
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)

URL_RE = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)
NUM_RE = re.compile(r"\b\d+(?:[.,]\d+)?\b", re.UNICODE)
SPACE_RE = re.compile(r"\s+", re.UNICODE)
WORD_RE = re.compile(r"^[a-zа-яё]+$", re.IGNORECASE)

stop_words = sorted(
    set(stopwords.words("russian"))
    | set(stopwords.words("english"))
    | {
        "__url__",
        "__num__",
        "afp",
        "associated",
        "bbc",
        "cnn",
        "daily",
        "news",
        "num",
        "of",
        "post",
        "press",
        "reuters",
        "ru",
        "это",
        "который",
        "которые",
        "очень",
    }
)


def normalize_text(text: str) -> str:
    text = text.lower()
    text = URL_RE.sub(" __url__ ", text)
    text = NUM_RE.sub(" __num__ ", text)
    return SPACE_RE.sub(" ", text).strip()


def lemmatize_text(text: str) -> str:
    try:
        from pymorphy3 import MorphAnalyzer
        from razdel import tokenize
    except ModuleNotFoundError:
        return "[лемматизация в этом окружении не запущена]"

    morph = MorphAnalyzer()
    normalized = normalize_text(text)
    parts = []
    for token in tokenize(normalized):
        value = token.text
        if WORD_RE.fullmatch(value) is None:
            parts.append(value)
            continue
        parts.append(morph.parse(value)[0].normal_form)
    return " ".join(parts)


preview_rows = sample_df.head(2).copy()
preview_rows["text_only_light"] = preview_rows["text"].map(normalize_text)
preview_rows["text_only_lemma"] = preview_rows["text"].map(lemmatize_text)
display(preview_rows[["title", "text_only_light", "text_only_lemma"]])



,title,text_only_light,text_only_lemma
0,Организатор теракта у итальянского колледжа пр...,"джованни вантаджато (giovanni vantaggiato), ус...","джованни вантаджать ( giovanni vantaggiato ) ,..."
1,Президент пообещал программу материнского капи...,президент россии владимир путин в ходе ежегодн...,президент россия владимир путин в ход ежегодны...


Нормализация убирает лишний шум из текстов. Лемматизация дополнительно схлопывает словоформы, но здесь уже есть возможный побочный эффект: трансформерные и гибридные модели часто выигрывают от более естественного текста.


In [4]:
structure_df = pd.DataFrame(results["structure"]["experiments"])
structure_admissible = structure_df[structure_df["admissible"]].copy()


def summarize_by(df: pd.DataFrame, field: str) -> pd.DataFrame:
    summary = (
        df.groupby(field)
        .agg(
            best_selection=("selection_score", "max"),
            mean_selection=("selection_score", "mean"),
            mean_coherence=("coherence_umass", "mean"),
            mean_diversity=("topic_diversity", "mean"),
            mean_coverage=("coverage", "mean"),
            mean_topic_count=("topic_count", "mean"),
        )
        .sort_values(["best_selection", "mean_selection"], ascending=False)
    )
    return summary.round(4)


best_embed_variants = (
    structure_admissible.sort_values("selection_score", ascending=False)
    .groupby("embed_text", as_index=False)
    .first()[
        [
            "embed_text",
            "vec_text",
            "encoder",
            "clustering",
            "dim_reduction",
            "topic_count",
            "topic_diversity",
            "coherence_umass",
            "selection_score",
        ]
    ]
    .rename(columns={"embed_text": "text_for_embeddings"})
    .sort_values("selection_score", ascending=False)
    .reset_index(drop=True)
)

best_vec_variants = (
    structure_admissible.sort_values("selection_score", ascending=False)
    .groupby("vec_text", as_index=False)
    .first()[
        [
            "vec_text",
            "embed_text",
            "encoder",
            "clustering",
            "dim_reduction",
            "topic_count",
            "topic_diversity",
            "coherence_umass",
            "selection_score",
        ]
    ]
    .rename(columns={"vec_text": "text_for_topic_words"})
    .sort_values("selection_score", ascending=False)
    .reset_index(drop=True)
)

print("Предобработка перед энкодером")
display(summarize_by(structure_admissible, "embed_text"))
display(best_embed_variants.round(4))
print("Предобработка перед токенизацией")
display(summarize_by(structure_admissible, "vec_text"))
display(best_vec_variants.round(4))


Предобработка перед энкодером


,best_selection,mean_selection,mean_coherence,mean_diversity,mean_coverage,mean_topic_count
embed_text,,,,,,
text_only_light,0.6950,0.5897,-14.9494,0.9770,1.0,17.5
title_text_light,0.6655,0.5680,-15.3400,0.9717,1.0,17.5
text_only_lemma,0.6631,0.5636,-15.4337,0.9715,1.0,17.5
title_text_lemma,0.6596,0.5678,-15.4175,0.9755,1.0,17.5


,text_for_embeddings,vec_text,encoder,clustering,dim_reduction,topic_count,topic_diversity,coherence_umass,selection_score
0,text_only_light,title_text_light,tfidf_svd,kmeans_15,umap_15,15,0.9800,-12.6226,0.6950
1,title_text_light,text_only_lemma,tfidf_svd,kmeans_20,umap_30,20,0.9900,-13.4848,0.6655
2,text_only_lemma,text_only_lemma,tfidf_svd,kmeans_15,umap_15,15,0.9667,-13.0872,0.6631
3,title_text_lemma,text_only_lemma,tfidf_svd,kmeans_20,umap_15,20,0.9900,-13.6196,0.6596


Предобработка перед токенизацией


,best_selection,mean_selection,mean_coherence,mean_diversity,mean_coverage,mean_topic_count
vec_text,,,,,,
title_text_light,0.6950,0.5706,-15.3462,0.9751,1.0,17.5
text_only_light,0.6810,0.5712,-15.3152,0.9742,1.0,17.5
title_text_lemma,0.6766,0.5734,-15.2669,0.9742,1.0,17.5
text_only_lemma,0.6757,0.5740,-15.2123,0.9722,1.0,17.5


,text_for_topic_words,embed_text,encoder,clustering,dim_reduction,topic_count,topic_diversity,coherence_umass,selection_score
0,title_text_light,text_only_light,tfidf_svd,kmeans_15,umap_15,15,0.9800,-12.6226,0.6950
1,text_only_light,text_only_light,tfidf_svd,kmeans_15,umap_15,15,0.9667,-12.6817,0.6810
2,title_text_lemma,text_only_light,tfidf_svd,kmeans_15,umap_15,15,0.9733,-12.9107,0.6766
3,text_only_lemma,text_only_light,tfidf_svd,kmeans_15,umap_30,15,0.9867,-13.1888,0.6757


На этапе построения document embeddings лучшим оказался `text_only_light`. Разница с лемматизированными вариантами не драматическая, но стабильна: у мягкой нормализации чуть выше лучшие значения `selection_score`, а по coherence она тоже не проседает.

Использовать только нормализацию перед токенизаций тоже получилось более выгодно по метрикам.

По входному тексту получилось, что подавать на первом этапе лучше текст без заголовков, а перед токенизацией с заголовками.

Note: это список лучших представителей для каждого варианта текста. Если в нескольких строках повторяется `tfidf_svd`, это значит, что именно он оказался лучшим внутри этих групп. Полное сравнение энкодеров приведено в следующем подразделе.



## 3. Подбор элементов пайплайна BERTopic

В этом разделе я иду по компонентам пайплайна по отдельности. При сравнении я опираюсь на результаты экспериментов из `outputs/bertopic_search/results.json`. Кроме метрик, я слежу за выполнением критериев: покрытие не ниже `0.65` и числом тем в диапазоне `12..40`.

### 3.1. Выбор энкодера



In [5]:
encoder_summary = summarize_by(structure_admissible, "encoder")
best_encoder_runs = (
    structure_admissible.sort_values("selection_score", ascending=False)
    .groupby("encoder", as_index=False)
    .first()[
        [
            "encoder",
            "embed_text",
            "vec_text",
            "clustering",
            "dim_reduction",
            "topic_count",
            "topic_diversity",
            "coherence_umass",
            "selection_score",
        ]
    ]
    .sort_values("selection_score", ascending=False)
    .reset_index(drop=True)
)

display(encoder_summary)
display(best_encoder_runs.round(4))





,best_selection,mean_selection,mean_coherence,mean_diversity,mean_coverage,mean_topic_count
encoder,,,,,,
tfidf_svd,0.6950,0.6191,-14.2194,0.9736,1.0,17.5
rubert,0.6271,0.5282,-16.1749,0.9683,1.0,17.5
minilm,0.6200,0.5696,-15.4612,0.9798,1.0,17.5


,encoder,embed_text,vec_text,clustering,dim_reduction,topic_count,topic_diversity,coherence_umass,selection_score
0,tfidf_svd,text_only_light,title_text_light,kmeans_15,umap_15,15,0.9800,-12.6226,0.6950
1,rubert,text_only_light,text_only_lemma,kmeans_15,umap_15,15,0.9933,-14.4210,0.6271
2,minilm,text_only_light,title_text_lemma,kmeans_15,umap_30,15,0.9933,-14.5801,0.6200


Здесь победитель оказался `TF-IDF + TruncatedSVD`. Он обходит обе sentence-transformer модели по лучшему и по среднему `selection_score`. Для этого корпуса такая картина выглядит правдоподобно. Новостные темы часто держатся на чётких лексических маркерах: названия стран, имена, организации, спортивных команд, судебных терминов. Лексический энкодер сохраняет эту структуру очень хорошо.

`MiniLM` и `RuBERT` я не выкидывал заранее. Они были в поиске наравне с остальными, но у них и coherence, и итоговый балл оказываются слабее. Поэтому дальше логично оставить `tfidf_svd`.



### 3.2. Выбор снижения размерности



In [6]:
dim_full_summary = (
    structure_df.groupby("dim_reduction")
    .agg(
        admissible_rate=("admissible", "mean"),
        best_selection=("selection_score", "max"),
        mean_selection=("selection_score", "mean"),
        mean_topic_count=("topic_count", "mean"),
        mean_coverage=("coverage", "mean"),
    )
    .sort_values(["best_selection", "mean_selection"], ascending=False)
    .round(4)
)
dim_admissible_summary = summarize_by(structure_admissible, "dim_reduction")

best_dim_runs = (
    structure_admissible.sort_values("selection_score", ascending=False)
    .groupby("dim_reduction", as_index=False)
    .first()[
        [
            "dim_reduction",
            "encoder",
            "clustering",
            "embed_text",
            "vec_text",
            "topic_count",
            "topic_diversity",
            "coherence_umass",
            "selection_score",
        ]
    ]
    .sort_values("selection_score", ascending=False)
    .reset_index(drop=True)
)

display(dim_full_summary)
display(dim_admissible_summary)
display(best_dim_runs.round(4))





,admissible_rate,best_selection,mean_selection,mean_topic_count,mean_coverage
dim_reduction,,,,,
pca_10,0.5,0.9238,0.5737,10.6875,0.7990
umap_15,0.5,0.6950,0.5167,80.3750,0.8451
umap_30,0.5,0.6803,0.5159,71.0625,0.8253


,best_selection,mean_selection,mean_coherence,mean_diversity,mean_coverage,mean_topic_count
dim_reduction,,,,,,
umap_15,0.6950,0.5931,-14.9435,0.9806,1.0,17.5
umap_30,0.6803,0.5916,-15.0077,0.9822,1.0,17.5
pca_10,0.6205,0.5321,-15.9042,0.9589,1.0,17.5


,dim_reduction,encoder,clustering,embed_text,vec_text,topic_count,topic_diversity,coherence_umass,selection_score
0,umap_15,tfidf_svd,kmeans_15,text_only_light,title_text_light,15,0.9800,-12.6226,0.6950
1,umap_30,tfidf_svd,kmeans_15,text_only_light,title_text_light,15,0.9867,-13.0864,0.6803
2,pca_10,tfidf_svd,kmeans_15,text_only_light,title_text_light,15,0.9733,-14.1809,0.6205


По полной таблице может показаться, что `PCA(10)` очень сильна, но это впечатление создают вырожденные конфигурации с `HDBSCAN`, которые схлопывали корпус в 2–4 крупные темы. Для тематического моделирования такой результат слишком грубый, поэтому в выборе я ориентируюсь на допустимые решения.

В admissible-зоне выигрывает `UMAP`, причём `n_neighbors=15` оказывается чуть лучше `30`. Для финальной модели я беру `UMAP`.



### 3.3. Выбор алгоритма кластеризации



In [7]:
clustering_full_summary = (
    structure_df.groupby("clustering")
    .agg(
        admissible_rate=("admissible", "mean"),
        best_selection=("selection_score", "max"),
        mean_selection=("selection_score", "mean"),
        mean_topic_count=("topic_count", "mean"),
        mean_coverage=("coverage", "mean"),
        mean_coherence=("coherence_umass", "mean"),
    )
    .sort_values(["best_selection", "mean_selection"], ascending=False)
    .round(4)
)

clustering_admissible_summary = summarize_by(structure_admissible, "clustering")

top_hdbscan_runs = (
    structure_df[structure_df["clustering"].str.startswith("hdbscan")]
    .sort_values("selection_score", ascending=False)
    .head(8)[
        [
            "experiment_id",
            "embed_text",
            "vec_text",
            "encoder",
            "dim_reduction",
            "topic_count",
            "coverage",
            "topic_diversity",
            "coherence_umass",
            "admissible",
        ]
    ]
)

display(clustering_full_summary)
display(clustering_admissible_summary)
print("Примеры лучших конфигураций с HDBSCAN, которые не подходят по критериям")
display(top_hdbscan_runs.round(4))



,admissible_rate,best_selection,mean_selection,mean_topic_count,mean_coverage,mean_coherence
clustering,,,,,,
hdbscan_25,0.0,0.9238,0.5240,61.3333,0.6465,-14.4490
hdbscan_10,0.0,0.8133,0.4732,119.8333,0.6460,-13.8407
kmeans_15,1.0,0.6950,0.5823,15.0000,1.0000,-15.0223
kmeans_20,1.0,0.6744,0.5623,20.0000,1.0000,-15.5480


,best_selection,mean_selection,mean_coherence,mean_diversity,mean_coverage,mean_topic_count
clustering,,,,,,
kmeans_15,0.6950,0.5823,-15.0223,0.9721,1.0,15.0
kmeans_20,0.6744,0.5623,-15.5480,0.9758,1.0,20.0


Примеры лучших конфигураций с HDBSCAN, которые не подходят по критериям


,experiment_id,embed_text,vec_text,encoder,dim_reduction,topic_count,coverage,topic_diversity,coherence_umass,admissible
359,struct_title_text_light_text_only_lemma_tfidf_...,title_text_light,text_only_lemma,tfidf_svd,pca_10,2,0.8995,1.0,-7.2590,False
431,struct_title_text_light_title_text_lemma_tfidf...,title_text_light,title_text_lemma,tfidf_svd,pca_10,2,0.8995,1.0,-7.5647,False
107,struct_text_only_light_title_text_light_tfidf_...,text_only_light,title_text_light,tfidf_svd,pca_10,2,0.9151,1.0,-7.6683,False
395,struct_title_text_light_title_text_light_tfidf...,title_text_light,title_text_light,tfidf_svd,pca_10,2,0.8995,1.0,-7.6270,False
323,struct_title_text_light_text_only_light_tfidf_...,title_text_light,text_only_light,tfidf_svd,pca_10,2,0.8995,1.0,-7.7020,False
143,struct_text_only_light_title_text_lemma_tfidf_...,text_only_light,title_text_lemma,tfidf_svd,pca_10,2,0.9151,1.0,-7.9264,False
71,struct_text_only_light_text_only_lemma_tfidf_s...,text_only_light,text_only_lemma,tfidf_svd,pca_10,2,0.9151,1.0,-8.1823,False
35,struct_text_only_light_text_only_light_tfidf_s...,text_only_light,text_only_light,tfidf_svd,pca_10,2,0.9151,1.0,-8.7003,False


`HDBSCAN` в этом корпусе чаще даёт две крайности. Часть конфигураций схлопывает тексты в несколько слишком широких тем, а часть, наоборот, дробит корпус на очень много маленьких кластеров и понижает покрытие.

`KMeans` ведёт себя гораздо устойчивее. У него покрытие всегда равно `1.0`, а число тем контролируется напрямую. На admissible-конфигурациях именно ветка `KMeans` даёт лучших кандидатов, поэтому в финал проходит она.

### 3.4. Выбор токенизации и словаря тем



In [8]:
peripherals_df = pd.DataFrame(results["peripherals"]["experiments"])
vectorizer_summary = summarize_by(peripherals_df, "vectorizer")
best_vectorizer_runs = (
    peripherals_df.sort_values("selection_score", ascending=False)
    .groupby("vectorizer", as_index=False)
    .first()[
        [
            "vectorizer",
            "embed_text",
            "vec_text",
            "representation",
            "topic_diversity",
            "coherence_umass",
            "selection_score",
            "sample_topic_words",
        ]
    ]
    .sort_values("selection_score", ascending=False)
    .reset_index(drop=True)
)

display(vectorizer_summary)
display(best_vectorizer_runs.round(4))





,best_selection,mean_selection,mean_coherence,mean_diversity,mean_coverage,mean_topic_count
vectorizer,,,,,,
bigram,0.9964,0.6510,-7.7778,0.9600,1.0,15.0
unigram,0.9812,0.6328,-8.0205,0.9578,1.0,15.0


,vectorizer,embed_text,vec_text,representation,topic_diversity,coherence_umass,selection_score,sample_topic_words
0,bigram,text_only_light,title_text_light,mmr,1.0000,-3.0475,0.9964,"уголовное, уголовное дело, возбуждено, возбужд..."
1,unigram,text_only_light,text_only_light,mmr,0.9933,-2.9741,0.9812,"мвд, возбуждено, происшествия, сотрудники, сле..."


Биграммы лучше удерживают устойчивые газетные формулы вроде `уголовное дело`, `президент россии`, `чемпионат мира`, `верховная рада`, `миллиарда долларов`. Для тематического моделирования новостей это особенно полезно: многие темы читаются именно через такие словосочетания.

Формально `bigram` выигрывает и по среднему, и по лучшему `selection_score`, поэтому дальше я оставляю его как базовый словарь для финального кандидата.



### 3.5. Выбор способа описания тем



In [9]:
vec_text_summary = summarize_by(peripherals_df, "vec_text")
representation_summary = summarize_by(peripherals_df, "representation")

vec_repr_pivot = (
    peripherals_df.pivot_table(
        index="representation",
        columns="vec_text",
        values="selection_score",
        aggfunc="max",
    )
    .round(4)
    .sort_index()
)

best_representation_runs = (
    peripherals_df.sort_values("selection_score", ascending=False)
    .groupby("representation", as_index=False)
    .first()[
        [
            "representation",
            "vec_text",
            "vectorizer",
            "topic_diversity",
            "coherence_umass",
            "selection_score",
            "sample_topic_words",
        ]
    ]
    .sort_values("selection_score", ascending=False)
    .reset_index(drop=True)
)

display(vec_text_summary)
display(vec_repr_pivot)
display(representation_summary)
display(best_representation_runs.round(4))





,best_selection,mean_selection,mean_coherence,mean_diversity,mean_coverage,mean_topic_count
vec_text,,,,,,
title_text_light,0.9964,0.6407,-7.9621,0.9596,1.0,15.0
text_only_light,0.9925,0.6442,-7.7731,0.9575,1.0,15.0


vec_text,text_only_light,title_text_light
representation,,
ctfidf,0.4916,0.5143
ctfidf_no_reduce,0.2906,0.3193
keybert,0.8643,0.8747
mmr,0.9925,0.9964


,best_selection,mean_selection,mean_coherence,mean_diversity,mean_coverage,mean_topic_count
representation,,,,,,
mmr,0.9964,0.9835,-3.1178,0.9967,1.0,15.0
keybert,0.8747,0.8448,-3.8413,0.9600,1.0,15.0
ctfidf,0.5143,0.4623,-12.6888,0.9789,1.0,15.0
ctfidf_no_reduce,0.3193,0.2769,-11.9486,0.9000,1.0,15.0


,representation,vec_text,vectorizer,topic_diversity,coherence_umass,selection_score,sample_topic_words
0,mmr,title_text_light,bigram,1.0000,-3.0475,0.9964,"уголовное, уголовное дело, возбуждено, возбужд..."
1,keybert,title_text_light,bigram,0.9667,-3.6143,0.8747,"автомобиль, автомобиля, агентства, области, ра..."
2,ctfidf,title_text_light,bigram,0.9867,-12.0757,0.5143,"riot, pussy riot, pussy, riot pussy, life, mer..."
3,ctfidf_no_reduce,title_text_light,bigram,0.9067,-11.4682,0.3193,"pussy riot, pussy, riot, life, riot pussy, mer..."


Для document embeddings лучшим был `text_only_light`, но для описания тем лучше использовать `title_text_light`, что довольно логично. Заголовок даёт компактный сюжетный сигнал, а основной текст добавляет детали. В паре с `CountVectorizer` и `c-TF-IDF` такой текст делает top words заметно содержательнее.

Среди representation-моделей уверенно выигрывает `MMR`. У него и coherence, и diversity заметно выше, чем у `c-TF-IDF`. Это хороший результат и по содержанию: `MMR` убирает лишние повторы и делает список слов более компактным, но при этом не теряет тему. `KeyBERTInspired` остаётся сильным запасным вариантом, однако в этой задаче он уступил `MMR`.

### 3.6. Подбор гиперпараметров финальной конфигурации



In [10]:
tuning_probes_df = pd.DataFrame(results["tuning"]["probes"])
tuning_combined_df = pd.DataFrame(results["tuning"]["combined"])
tuning_finalists_df = pd.DataFrame(results["tuning"]["finalists"])
confirmation_df = pd.DataFrame(results["confirmation"]["summary"])

top_probe_rows = tuning_probes_df.sort_values("selection_score", ascending=False).head(12)[
    [
        "experiment_id",
        "encoder",
        "dim_reduction",
        "clustering",
        "vectorizer",
        "representation",
        "topic_count",
        "topic_diversity",
        "coherence_umass",
        "selection_score",
    ]
]

display(top_probe_rows.round(4))
display(tuning_combined_df.round(4))
display(confirmation_df.round(4))



,experiment_id,encoder,dim_reduction,clustering,vectorizer,representation,topic_count,topic_diversity,coherence_umass,selection_score
5,tuneprobe_peri_struct_text_only_light_title_te...,tfidf_svd,umap_15,kmeans_40,bigram,mmr,40,0.9975,-2.5965,0.9047
32,tuneprobe_peri_struct_text_only_light_title_te...,tfidf_svd_128,umap_30,kmeans_15,bigram,mmr,15,0.9933,-2.3751,0.9000
12,tuneprobe_peri_struct_text_only_light_title_te...,tfidf_svd_128,umap_15,kmeans_15,bigram,mmr,15,1.0000,-2.8454,0.8771
41,tuneprobe_peri_struct_text_only_light_text_onl...,tfidf_svd,umap_15,kmeans_20,bigram,mmr,20,1.0000,-2.8830,0.8673
45,tuneprobe_peri_struct_text_only_light_text_onl...,tfidf_svd,umap_15,kmeans_40,bigram,mmr,40,0.9975,-2.7497,0.8646
3,tuneprobe_peri_struct_text_only_light_title_te...,tfidf_svd,umap_15,kmeans_30,bigram,mmr,30,1.0000,-2.9231,0.8568
47,tuneprobe_peri_struct_text_only_light_text_onl...,tfidf_svd,umap_nn20_c5,kmeans_15,bigram,mmr,15,1.0000,-2.9278,0.8556
52,tuneprobe_peri_struct_text_only_light_text_onl...,tfidf_svd_128,umap_15,kmeans_15,bigram,mmr,15,1.0000,-2.9554,0.8484
56,tuneprobe_peri_struct_text_only_light_text_onl...,tfidf_svd,umap_15,kmeans_15,uni_df2_max95,mmr,15,1.0000,-2.9962,0.8377
4,tuneprobe_peri_struct_text_only_light_title_te...,tfidf_svd,umap_15,kmeans_35,bigram,mmr,35,0.9943,-2.7135,0.8259


,seed,experiment_id,embed_text,vec_text,encoder,dim_reduction,clustering,vectorizer,representation,elapsed_sec,...,phase,parent_experiment_id,admissible,coherence_umass_norm,topic_diversity_norm,coverage_norm,selection_score,candidate_kind,applied_variants,_component_configs
0,42,baselinecarry_peri_struct_text_only_light_titl...,text_only_light,title_text_light,tfidf_svd,umap_15,kmeans_15,bigram,mmr,8.02,...,tuning_combined,peri_struct_text_only_light_title_text_light_t...,True,0.1460,1.0,1.0,0.5730,baseline,baseline,None
1,42,tuned_peri_struct_text_only_light_title_text_l...,text_only_light,title_text_light,tfidf_svd_128,umap_nn15_c5,kmeans_40,bi_df2_max95,mmr,8.43,...,tuning_combined,peri_struct_text_only_light_title_text_light_t...,True,1.0000,0.0,1.0,0.7000,tuned,clustering=kmeans_40; dim_reduction=umap_nn15_...,"{""clustering"": {""type"": ""kmeans"", ""n_clusters""..."
2,42,baselinecarry_peri_struct_text_only_light_titl...,text_only_light,title_text_light,tfidf_svd,umap_30,kmeans_15,bigram,mmr,10.09,...,tuning_combined,peri_struct_text_only_light_title_text_light_t...,True,0.1369,1.0,1.0,0.5685,baseline,baseline,None
3,42,tuned_peri_struct_text_only_light_title_text_l...,text_only_light,title_text_light,tfidf_svd,umap_nn15_c5,kmeans_40,bigram,mmr,8.75,...,tuning_combined,peri_struct_text_only_light_title_text_light_t...,True,0.9760,0.5,1.0,0.8380,tuned,clustering=kmeans_40; dim_reduction=umap_nn15_c5,"{""clustering"": {""type"": ""kmeans"", ""n_clusters""..."
4,42,baselinecarry_peri_struct_text_only_light_text...,text_only_light,text_only_light,tfidf_svd,umap_15,kmeans_15,bigram,mmr,7.88,...,tuning_combined,peri_struct_text_only_light_text_only_light_tf...,True,0.0000,1.0,1.0,0.5000,baseline,baseline,None
5,42,tuned_peri_struct_text_only_light_text_only_li...,text_only_light,text_only_light,tfidf_svd_128,umap_nn20_c5,kmeans_40,uni_df2_max95,mmr,7.48,...,tuning_combined,peri_struct_text_only_light_text_only_light_tf...,True,0.8465,0.0,1.0,0.6232,tuned,clustering=kmeans_40; dim_reduction=umap_nn20_...,"{""clustering"": {""type"": ""kmeans"", ""n_clusters""..."


,experiment_id,embed_text,vec_text,encoder,dim_reduction,clustering,vectorizer,representation,topic_count,std_topic_count,...,mean_nmi_topic,std_nmi_topic,mean_ari_topic,std_ari_topic,mean_elapsed_sec,admissible,coherence_umass_norm,topic_diversity_norm,coverage_norm,selection_score
0,tuned_peri_struct_text_only_light_text_only_li...,text_only_light,text_only_light,tfidf_svd_128,umap_nn20_c5,kmeans_40,uni_df2_max95,mmr,40.0,0.0,...,0.3716,0.0019,0.1398,0.0041,7.4800,True,1.0000,0.0,1.0,0.7000
1,tuned_peri_struct_text_only_light_title_text_l...,text_only_light,title_text_light,tfidf_svd_128,umap_nn15_c5,kmeans_40,bi_df2_max95,mmr,40.0,0.0,...,0.3718,0.0058,0.1452,0.0086,8.5167,True,0.9751,0.2,1.0,0.7476
2,tuned_peri_struct_text_only_light_title_text_l...,text_only_light,title_text_light,tfidf_svd,umap_nn15_c5,kmeans_40,bigram,mmr,40.0,0.0,...,0.3816,0.0029,0.1628,0.0044,8.7767,True,0.0000,1.0,1.0,0.5000


На этапе тонкой настройки выигрыш пришёл сразу с нескольких сторон. Лучший кандидат перешёл на `tfidf_svd_128`, `KMeans(40)`, `UMAP(n_neighbors=15, n_components=5)` и `CountVectorizer` с `ngram_range=(1, 2)`, `min_df=2`, `max_df=0.95`.

После повторного прогона на трёх seed-ах победителем осталась конфигурация:

- embeddings: `text_only_light -> TF-IDF + TruncatedSVD(128)`;
- topic vocabulary: `title_text_light -> CountVectorizer(1,2; min_df=2; max_df=0.95)`;
- dimensionality reduction: `UMAP(15, 5)`;
- clustering: `MiniBatchKMeans(40)`;
- topic representation: `MMR`.

Вопрос про `40` тем я оставляю до раздела с финальной моделью. Там уже можно смотреть на сами темы целиком, а не только на таблицу с метриками.



## 4. Обучение финальной модели по шагам

Ниже я заново обучаю выбранный пайплайн внутри ноутбука. Эта часть не зависит от экспериментальных скриптов: все нужные функции и параметры объявлены здесь.



In [11]:
import gc
import random
import warnings

import plotly.express as px
import plotly.graph_objects as go
import torch
from bertopic import BERTopic
from bertopic.backend._sklearn import SklearnEmbedder
from bertopic.representation import MaximalMarginalRelevance
from bertopic.vectorizers import ClassTfidfTransformer
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.pipeline import make_pipeline
from umap import UMAP

warnings.filterwarnings("ignore", category=FutureWarning)


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass


def build_final_embeddings(texts: list[str], seed: int = SEED) -> tuple[np.ndarray, SklearnEmbedder]:
    embedding_pipe = make_pipeline(
        TfidfVectorizer(
            stop_words=stop_words,
            ngram_range=(1, 2),
            min_df=5,
            max_df=0.8,
            max_features=50_000,
            sublinear_tf=True,
            token_pattern=r"(?u)\b[\wёЁ-]+\b",
        ),
        TruncatedSVD(n_components=128, random_state=seed),
    )
    embeddings = embedding_pipe.fit_transform(texts).astype("float32")
    return embeddings, SklearnEmbedder(embedding_pipe)


def compute_topic_metrics(
    topic_model: BERTopic,
    texts: list[str],
    topics: list[int],
    true_labels: list[str],
    top_n_words: int = 10,
) -> dict:
    topic_ids = sorted(t for t in set(topics) if t != -1)
    coverage = float(np.mean(np.asarray(topics) != -1))

    topic_words = []
    for topic_id in topic_ids:
        words = [word for word, _ in topic_model.get_topic(topic_id)[:top_n_words]]
        if words:
            topic_words.append(words)

    unique_words = len({word for words in topic_words for word in words})
    total_words = sum(len(words) for words in topic_words)
    topic_diversity = unique_words / total_words if total_words else 0.0

    analyzer = topic_model.vectorizer_model.build_analyzer()
    tokenized_docs = [analyzer(text) for text in texts]
    dictionary = Dictionary(tokenized_docs)
    corpus = [dictionary.doc2bow(doc) for doc in tokenized_docs]

    filtered_topic_words = []
    for words in topic_words:
        valid = [word for word in words if word in dictionary.token2id]
        if len(valid) >= 2:
            filtered_topic_words.append(valid)

    coherence_umass = float("nan")
    if filtered_topic_words:
        coherence_umass = float(
            CoherenceModel(
                topics=filtered_topic_words,
                corpus=corpus,
                dictionary=dictionary,
                coherence="u_mass",
            ).get_coherence()
        )

    return {
        "topic_count": len(topic_ids),
        "coverage": coverage,
        "topic_diversity": topic_diversity,
        "coherence_umass": coherence_umass,
        "nmi_topic": float(normalized_mutual_info_score(true_labels, topics)),
        "ari_topic": float(adjusted_rand_score(true_labels, topics)),
    }



In [12]:
final_embed_texts = sample_df["text"].map(normalize_text).tolist()
final_vec_texts = (
    sample_df["title"].fillna("").map(normalize_text)
    + " "
    + sample_df["text"].map(normalize_text)
).str.strip().tolist()

final_embeddings, representation_backend = build_final_embeddings(final_embed_texts, seed=SEED)
final_embeddings.shape



(12000, 128)

На этом шаге я использую только нормализованный основной текст. Для лексического энкодера этого оказалось достаточно. `TruncatedSVD(128)` оставляет компактное представление документа и по результатам tuning оказался лучшим компромиссом между качеством и размерностью.

In [13]:
final_vectorizer = CountVectorizer(
    stop_words=stop_words,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    token_pattern=r"(?u)\b[\wёЁ-]+\b",
)

final_topic_model = BERTopic(
    embedding_model=representation_backend,
    umap_model=UMAP(
        n_neighbors=15,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=SEED,
        transform_seed=SEED,
    ),
    hdbscan_model=MiniBatchKMeans(
        n_clusters=40,
        random_state=SEED,
        batch_size=1024,
        n_init="auto",
    ),
    vectorizer_model=final_vectorizer,
    ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
    representation_model=MaximalMarginalRelevance(diversity=0.3),
    top_n_words=10,
    calculate_probabilities=False,
    verbose=False,
)

final_topics, _ = final_topic_model.fit_transform(final_vec_texts, embeddings=final_embeddings)
final_metrics = compute_topic_metrics(
    final_topic_model,
    final_vec_texts,
    final_topics,
    sample_df["topic"].tolist(),
    top_n_words=10,
)

final_metrics_df = pd.DataFrame([final_metrics]).round(4)
topic_info = final_topic_model.get_topic_info()
display(final_metrics_df)
display(topic_info.head(10))



,topic_count,coverage,topic_diversity,coherence_umass,nmi_topic,ari_topic
0,40,1.0,0.9975,-2.4023,0.3666,0.1444


,Topic,Count,Name,Representation,Representative_Docs
0,0,617,0_годам_лишения_приговор_виновным,"[годам, лишения, приговор, виновным, годам лиш...",[мосгорсуд решил оставить платона лебедева в с...
1,1,549,1_команды_матче_клуба_лиги,"[команды, матче, клуба, лиги, матча, очков, за...",[сборная испании открыла счет в матче с россие...
2,2,526,2_акций_миллиарда долларов_ммвб_процента акций,"[акций, миллиарда долларов, ммвб, процента акц...",[обзор рынков: цена на нефть рухнула на девять...
3,3,517,3_уголовное дело_дело_возбуждено_возбуждено уг...,"[уголовное дело, дело, возбуждено, возбуждено ...",[фсб занялась делами арестованного офицера скр...
4,4,508,4_google_apple_сервис_представила,"[google, apple, сервис, представила, nokia, an...",[российские ретейлеры одежды начали «черную пя...
5,5,497,5_буш_трамп_обама_президента сша,"[буш, трамп, обама, президента сша, президент ...",[кремль смутила идея о празднике по поводу воз...
6,6,446,6_мчс_погибли_пострадавших_происшествия,"[мчс, погибли, пострадавших, происшествия, про...",[в доме престарелых в костромской области прои...
7,7,445,7_украины_порошенко_верховной_украине,"[украины, порошенко, верховной, украине, прези...",[аваков приструнил олланда за его позицию по д...
8,8,420,8_ребенка_врачи_отец_женщины,"[ребенка, врачи, отец, женщины, летняя, летней...",[дания отправляет научную экспедицию к северно...
9,9,411,9_полиции_инцидент_инцидент произошел_мужчина,"[полиции, инцидент, инцидент произошел, мужчин...",[житель штата вашингтон убил детей из-за ухода...


Финальная модель воспроизводит тот же тип решения, который победил в экспериментах: `40` тем, полное покрытие корпуса и сильный coherence для набора тем такого размера. 

LLM-названий я ставлю фиксированный seed и greedy decoding; на `mps` этого обычно достаточно для стабильности, хотя самая строгая воспроизводимость была бы на `cpu`.


In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer

QWEN_MODEL_NAME = "Qwen/Qwen3.5-0.8B"
qwen_device = "mps" if torch.backends.mps.is_available() else "cpu"
torch.manual_seed(SEED)
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME, local_files_only=True)
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_NAME,
    local_files_only=True,
    dtype=torch.float16 if qwen_device == "mps" else torch.float32,
)
qwen_model.to(qwen_device)
qwen_model.eval()

analysis_for_names = sample_df.copy()
analysis_for_names["Topic"] = final_topics


def normalize_generated_name(name: str) -> str:
    cleaned = " ".join(name.strip().split()).strip(" .,;:!?").strip("'").strip(chr(34)) #char(34) is "
    if not cleaned:
        return cleaned
    return cleaned[0].upper() + cleaned[1:]


def generate_topic_name(topic_words: list[str], representative_titles: list[str]) -> str:
    title_block = "\n".join(f"- {title}" for title in representative_titles)
    messages = [
        {
            "role": "system",
            "content": (
                "Name the news-cluster topic."
                "Answer: 2-4 words, noun phrase only, no verb, no quotation marks, no colon. "
                "A general topic name based on the keywords and sample titles. "
                "Be general, only topic name, no other text."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Top words: {', '.join(topic_words)}\n"
                f"Sample titles:\n{title_block}"
                f"Be general, output only topic name, no other text:"
            ),
        },
    ]
    prompt = qwen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_tokenizer(prompt, return_tensors="pt").to(qwen_device)
    generated = qwen_model.generate(
        **inputs,
        max_new_tokens=8,
        do_sample=False,
        num_beams=1,
        pad_token_id=qwen_tokenizer.eos_token_id,
    )
    text = qwen_tokenizer.decode(
        generated[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    cleaned = normalize_generated_name(text)
    fallback = normalize_generated_name(" ".join(topic_words[:3]))
    return cleaned or fallback


topic_name_map = {}
for topic_id in sorted(t for t in set(final_topics) if t != -1):
    words = [word for word, _ in final_topic_model.get_topic(topic_id)[:30]]
    subset = analysis_for_names[analysis_for_names["Topic"] == topic_id]
    representative_titles = subset["title"].dropna().sample(frac=1, random_state=SEED).head(20).tolist()
    topic_name_map[topic_id] = generate_topic_name(words, representative_titles)

topic_labels_preview = pd.DataFrame(
    {
        "Topic": sorted(topic_name_map),
        "LLM_name": [topic_name_map[topic_id] for topic_id in sorted(topic_name_map)],
    }
)
display(topic_labels_preview)

del qwen_model
gc.collect()
if qwen_device == "mps" and hasattr(torch, "mps") and hasattr(torch.mps, "empty_cache"):
    torch.mps.empty_cache()



The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

,Topic,LLM_name
0,0,Судебная защита
1,1,Хоккей
2,2,Рыночные акции
3,3,Уголовное дело
4,4,Google и Apple
5,5,Президент сша
6,6,Пожарные происшествия
7,7,Украинская власть
8,8,Военная война
9,9,Полиция


In [15]:
topic_rows = []
representative_docs = final_topic_model.get_representative_docs()
analysis_df = sample_df.copy()
analysis_df["Topic"] = final_topics

for topic_id in sorted(t for t in set(final_topics) if t != -1):
    words = [word for word, _ in final_topic_model.get_topic(topic_id)[:8]]
    subset = analysis_df[analysis_df["Topic"] == topic_id]
    dominant_counts = subset["topic"].value_counts()
    dominant_label = dominant_counts.index[0]
    dominant_share = dominant_counts.iloc[0] / len(subset)
    topic_rows.append(
        {
            "Topic": topic_id,
            "LLM_name": topic_name_map[topic_id],
            "Count": len(subset),
            "Dominant_lenta_topic": dominant_label,
            "Dominant_share": dominant_share,
            "Unique_lenta_topics": subset["topic"].nunique(),
            "Top_words": ", ".join(words),
            "Representative_title": subset.iloc[0]["title"],
            "Sample_text": final_vec_texts[subset.index[0]],
            "Representative_doc": (representative_docs.get(topic_id, [""])[0] if representative_docs.get(topic_id) else ""),
        }
    )

topic_summary_df = pd.DataFrame(topic_rows).sort_values("Topic").reset_index(drop=True)

dominant_label_summary = (
    topic_summary_df.groupby("Dominant_lenta_topic")
    .agg(
        Topics=("Topic", "count"),
        Documents=("Count", "sum"),
        Mean_dominant_share=("Dominant_share", "mean"),
    )
    .sort_values(["Topics", "Documents"], ascending=False)
    .round(4)
)

display(topic_summary_df[["Topic", "LLM_name", "Count", "Dominant_lenta_topic", "Dominant_share", "Top_words"]].head(15))
display(dominant_label_summary)



,Topic,LLM_name,Count,Dominant_lenta_topic,Dominant_share,Top_words
0,0,Судебная защита,617,Россия,0.502431,"годам, лишения, приговор, виновным, годам лише..."
1,1,Хоккей,549,Спорт,0.976321,"команды, матче, клуба, лиги, матча, очков, заб..."
2,2,Рыночные акции,526,Экономика,0.718631,"акций, миллиарда долларов, ммвб, процента акци..."
3,3,Уголовное дело,517,Россия,0.640232,"уголовное дело, дело, возбуждено, возбуждено у..."
4,4,Google и Apple,508,Интернет и СМИ,0.352362,"google, apple, сервис, представила, nokia, and..."
5,5,Президент сша,497,Мир,0.686117,"буш, трамп, обама, президента сша, президент с..."
6,6,Пожарные происшествия,446,Мир,0.466368,"мчс, погибли, пострадавших, происшествия, прои..."
7,7,Украинская власть,445,Бывший СССР,0.773034,"украины, порошенко, верховной, украине, презид..."
8,8,Военная война,420,Из жизни,0.302381,"ребенка, врачи, отец, женщины, летняя, летней,..."
9,9,Полиция,411,Мир,0.464720,"полиции, инцидент, инцидент произошел, мужчина..."


,Topics,Documents,Mean_dominant_share
Dominant_lenta_topic,,,
Мир,11,2617,0.5909
Россия,8,3089,0.5484
Экономика,4,1449,0.6868
Спорт,4,1002,0.9751
Культура,4,996,0.6657
Наука и техника,3,901,0.7165
Интернет и СМИ,2,760,0.3686
Бывший СССР,2,554,0.7030
Из жизни,1,420,0.3024


Теперь на темы можно смотреть по автоматически сгенерированным названиям и по спискам top words. Названия от Qwen 3.5 я использую как компактные подписи для таблиц и графиков. Основой интерпретации остаются сами топ-слова, representative documents и распределение исходных рубрик внутри найденных кластеров.

По итоговой таблице `40` тем выглядят содержательно. Внутри крупных рубрик корпуса, наши темы распадается на более узкие сюжетные блоки:

- внутри `Россия` отдельно выделяются судебные приговоры, протестные акции, пожары и катастрофы, политика, теракты;
- внутри `Мир` расходятся политика США, Украина, Северная Корея, иранская ядерная программа, авиаперевозки;
- внутри `Спорт` хорошо отделяются футбол, сборная России, бокс и теннис;
- внутри экономики читаются разные сюжеты: курс рубля, бюджет и госрасходы, биржевые сделки и газовые поставки.


## 5. Визуализация результатов



In [25]:
topic_tokens_table = topic_summary_df[["Topic", "LLM_name", "Count", "Top_words"]].copy()
display(topic_tokens_table)

,Topic,LLM_name,Count,Top_words
0,0,Судебная защита,617,"годам, лишения, приговор, виновным, годам лише..."
1,1,Хоккей,549,"команды, матче, клуба, лиги, матча, очков, заб..."
2,2,Рыночные акции,526,"акций, миллиарда долларов, ммвб, процента акци..."
3,3,Уголовное дело,517,"уголовное дело, дело, возбуждено, возбуждено у..."
4,4,Google и Apple,508,"google, apple, сервис, представила, nokia, and..."
5,5,Президент сша,497,"буш, трамп, обама, президента сша, президент с..."
6,6,Пожарные происшествия,446,"мчс, погибли, пострадавших, происшествия, прои..."
7,7,Украинская власть,445,"украины, порошенко, верховной, украине, презид..."
8,8,Военная война,420,"ребенка, врачи, отец, женщины, летняя, летней,..."
9,9,Полиция,411,"полиции, инцидент, инцидент произошел, мужчина..."


Уже по таблице видно, что темы в целом читаются устойчиво. Наиболее аккуратные блоки получились у спорта, культуры, науки и финансовых новостей. Более широкими выглядят темы, связанные с внутренней политикой и происшествиями в России; для такого корпуса это ожидаемо, потому что внутри этих рубрик сюжетный спектр очень широк.



In [26]:
topic_barchart = final_topic_model.visualize_barchart(
    top_n_topics=20,
    n_words=10,
    title="Топ-токены для двадцати самых крупных тем",
)
topic_barchart

На bar chart видно, что словари крупных тем читаются вполне устойчиво. У судебных и криминальных тем среди топ-слов держатся связки вроде `уголовное дело`, `приговор`, `заключен`. У спортивных тем хорошо собираются маркеры матчей и турниров, у экономических: `рубль`, `доллар`, `процент`, `миллиард`, у авиационной темы: `самолет`, `авиакомпания`, `посадка`. Повторов между темами немного, поэтому `MMR` здесь действительно улучшил читаемость топ-слов и сделал названия тем понятнее.


In [27]:
viz_umap = UMAP(
    n_neighbors=15,
    n_components=2,
    min_dist=0.0,
    metric="cosine",
    random_state=SEED,
    transform_seed=SEED,
)
embedding_2d = viz_umap.fit_transform(final_embeddings)

plot_df = pd.DataFrame(
    {
        "x": embedding_2d[:, 0],
        "y": embedding_2d[:, 1],
        "Topic": final_topics,
        "Topic_name": [topic_name_map.get(topic_id, "Outlier") for topic_id in final_topics],
        "Lenta_topic": sample_df["topic"],
        "Title": sample_df["title"],
    }
)

scatter = px.scatter(
    plot_df,
    x="x",
    y="y",
    color="Topic_name",
    hover_data=["Topic", "Lenta_topic", "Title"],
    opacity=0.55,
    width=1400,
    height=1000,
    title="Документы и найденные темы в 2D проекции",
)
scatter.update_layout(showlegend=False)

topic_sizes = plot_df.groupby("Topic", as_index=False).size().rename(columns={"size": "Count"})
centroids = (
    plot_df.groupby(["Topic", "Topic_name"], as_index=False)[["x", "y"]]
    .mean()
    .merge(topic_sizes, on="Topic", how="left")
    .sort_values(["Count", "Topic"], ascending=[False, True])
)
for _, row in centroids.head(24).iterrows():
    scatter.add_annotation(
        x=row["x"],
        y=row["y"],
        text=row["Topic_name"],
        showarrow=False,
        font=dict(size=10, color="black"),
        bgcolor="rgba(255,255,255,0.6)",
    )

scatter


![topics](../assets/hw_4/topics.png)

После просмотра карты видно, что пространство устроено неравномерно. Справа почти изолирован спортивный блок с футбольными матчами и сборной России. Отдельными компактными островами лежат наука и космос, авиаперевозки, вооружение и испытания, кино и режиссеры. Политические и криминальные темы собираются в более плотное центральное облако и частично перекрываются. Именно этот рисунок хорошо объясняет умеренные `NMI` и `ARI`: модель различает крупные сюжетные острова уверенно, а внутри новостного ядра темы естественно ближе друг к другу.


In [28]:
topic_distr, topic_token_distr = final_topic_model.approximate_distribution(
    final_vec_texts, calculate_tokens=True, batch_size=100, min_similarity=0.05
)
df = final_topic_model.visualize_approximate_distribution(final_vec_texts[11], topic_token_distr[11])
df

,отремонтировать,трубопроводы,газпрому,помогут,частники,газпром,разработал,новый,механизм,доступа,независимых,производителей,к,трубопроводу,в,результате,независимые,компании,будут,участвовать,в,реконструкции,трубопроводов,принадлежащих,монополисту,об,этом,пишут,ведомости,в,настоящий,момент,независимые,производители,топлива,договариваются,с,клиентом,о,покупке,затем,они,подают,в,монополию,заявку,с,просьбой,разрешить,прокачку,топлива,до,потребителя,за,прокачку,газа,компании,платят,__num__,доллара,за,__num__,кубометров,на,__num__,километров,согласно,новому,механизму,доступа,независимых,производителей,к,трубопроводу,компании,будут,платить,заранее,за,весь,объем,газа,который,собираются,прокачать,при,этом,в,расчет,не,принимается,тот,факт,удастся,независимым,производителям,продать,его,или,нет,кроме,того,газпром,намерен,привлечь,компании,к,реконструкции,и,строительству,трубопроводов,которые,будут,принадлежать,монополисту,газпром,также,введет,надбавку,к,тарифу,на,транспортировку,который,будет,согласован,с,федеральной,службой,по,тарифам,помимо,прочего,новый,механизм,доступа,предусматривает,кредитование,монополиста,независимыми,производителями,говорится,в,сообщении,газпрома,по,данным,издания,на,реконструкцию,газотранспортной,системы,газпрому,требуются,сотни,миллиардов,долларов,однако,в,бюджете,компании,__num__,года,на,инвестиции,в,транспортировку,заложено,всего,__num__,миллиардов,долларов,в,__num__,году,компания,вложит,в,эту,область,__num__,миллиардов,долларов,а,в,__num__,__num__,миллиардов
30_нефти_кубометров_газпром_газ,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.064,0.064,0.064,0.064,0.000,0.000,0.071,0.132,0.194,0.250,0.179,0.118,0.056,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.051,0.051,0.051,0.051,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.051,0.051,0.051,0.051,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000


In [29]:
df = final_topic_model.visualize_approximate_distribution(final_vec_texts[29], topic_token_distr[29])
df

,тренер,сборной,украины,объяснил,отсутствие,в,команде,игроков,из,рфпл,главный,тренер,сборной,украины,по,футболу,михаил,фоменко,рассказал,почему,решил,не,вызывать,в,команду,евгения,селезнева,и,александра,зинченко,из,кубани,и,уфы,об,этом,сообщает,football,ua,по,итогам,последнего,тура,в,рпл,селезнев,получил,негативные,оценки,специалистов,зинченко,перспективный,футболист,его,вызвали,в,молодежную,сборную,но,он,приболел,в,ближайшем,будущем,в,национальной,команде,будет,много,молодых,футболистов,рассказал,фоменко,в,четверг,__num__,марта,сборная,украины,на,своем,поле,в,товарищеском,матче,с,минимальным,счетом,обыграла,команду,кипра,__num__,__num__,в,среду,__num__,марта,стало,известно,что,зинченко,без,объяснения,причин,был,отчислен,из,состава,молодежной,сборной,украины,в,федерации,футбола,украины,ффу,это,объяснили,простудой,по,информации,чемпионат,com,ситуация,с,зинченко,возникла,из-за,напряженных,отношений,между,украиной,и,россией,__num__,марта,в,ффу,заявили,что,отсутствие,среди,вызванных,в,национальную,сборную,футболистов,выступающих,в,рфпл,обусловлено,только,спортивными,причинами
1_команды_матче_клуба_лиги,0.052,0.052,0.052,0.052,0.000,0.000,0.000,0.000,0.056,0.124,0.191,0.243,0.187,0.119,0.053,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
20_сборной_сборная_сборной россии_чемпионата мира,0.052,0.052,0.052,0.052,0.000,0.000,0.000,0.000,0.000,0.059,0.121,0.173,0.173,0.114,0.052,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
38_шарапова_open_теннисистка_сафина,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.053,0.053,0.053,0.053,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000


In [30]:
df = final_topic_model.visualize_approximate_distribution(final_vec_texts[80], topic_token_distr[80])
df

,лоуренс,фишберн,озвучит,серебряного,серфера,лоуренс,фишберн,озвучит,серебряного,серфера,в,фильме,фантастическая,четверка,вторжение,серебряного,серфера,сообщает,the,hollywood,reporter,со,ссылкой,на,создателей,картины,таким,образом,опровергнут,слух,о,том,что,голосом,фишберна,будет,говорить,другой,персонаж,галактус,студийная,запись,начнется,уже,с,__num__,апреля,а,сам,фильм,находится,на,момент,написания,данной,статьи,на,стадии,пост-продакшен,самого,серебряного,серфера,играет,дуг,джонс,однако,в,картине,его,образ,будет,полностью,заменен,компьютерным,созданным,студией,питера,джексона,weta,digital,в,фильме,также,снимались,майкл,чилкис,ян,гриффит,крис,эванс,и,джессика,альба,премьера,второй,части,фантастической,четверки,состоится,в,лондоне,__num__,июня,а,в,широкий,прокат,фильм,будет,выпущен,__num__,июня
14_фильм_режиссер_фильме_роль,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.053,0.106,0.159,0.159,0.106,0.053,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000


In [31]:
target_labels = ["Россия", "Спорт", "Экономика"]
selected_examples = (
    topic_summary_df[topic_summary_df["Dominant_lenta_topic"].isin(target_labels)]
    .sort_values(["Dominant_lenta_topic", "Dominant_share", "Count"], ascending=[True, False, False])
    .groupby("Dominant_lenta_topic", as_index=False)
    .first()[["Topic", "LLM_name", "Dominant_lenta_topic", "Sample_text"]]
    .reset_index(drop=True)
)

# Для токеновой heatmap я использую локальные окна вокруг каждого токена.
# BERTopic предсказывает тему для окна, после чего видно, где по тексту
# начинает доминировать соседний сюжетный блок.
WINDOW_SIZE = 8
MAX_TOKENS_ON_HEATMAP = 60
selected_examples


,Topic,LLM_name,Dominant_lenta_topic,Sample_text
0,11,Терракорты,Россия,организатор теракта у итальянского колледжа пр...
1,35,Бокс,Спорт,бывший чемпион мира по боксу победил цыпленка ...
2,30,Нефтегазовая индустрия,Экономика,"отремонтировать трубопроводы ""газпрому"" помогу..."


In [32]:
for row_number, row in selected_examples.iterrows():
    tokens = row["Sample_text"].split()
    token_count = min(len(tokens), MAX_TOKENS_ON_HEATMAP)
    tokens = tokens[:token_count]

    windows = []
    for token_idx in range(token_count):
        left = max(0, token_idx - WINDOW_SIZE // 2)
        right = min(token_count, token_idx + WINDOW_SIZE // 2 + 1)
        windows.append(" ".join(tokens[left:right]))

    window_topics, _ = final_topic_model.transform(windows)
    active_topics = pd.Series(window_topics).value_counts().head(4).index.tolist()
    token_matrix = np.zeros((len(active_topics), token_count), dtype=float)
    for token_idx, window_topic in enumerate(window_topics):
        if window_topic in active_topics:
            topic_position = active_topics.index(window_topic)
            token_matrix[topic_position, token_idx] = 1.0

    heatmap = go.Figure(
        data=go.Heatmap(
            z=token_matrix,
            x=tokens,
            y=[topic_name_map.get(int(topic_id), str(topic_id)) for topic_id in active_topics],
            colorscale="Blues",
            zmin=0.0,
            zmax=1.0,
            colorbar_title="hit",
        )
    )
    heatmap.update_layout(
        title=f"Тема {row['Topic']} | {row['Dominant_lenta_topic']} | {row['LLM_name']}",
        width=1600,
        height=600,
        xaxis_title="Токены текста",
        yaxis_title="Наиболее активные темы",
    )
    heatmap.update_xaxes(tickangle=60)
    display(heatmap)


![heatplot1](../assets/hw_4/heatplot1.png)
![heatplot2](../assets/hw_4/heatplot2.png)
![heatplot3](../assets/hw_4/heatplot3.png)

Эти heatmap уже полезно читать по конкретным примерам. В тексте из рубрики `Россия` доминирует тема терактов в Дагестане, но по ходу сообщения заметны локальные переключения к судебным приговорам, пожарам и политике Москвы. У спортивного примера ядро держится на боксе, при этом рядом всплывают футбольные и околокультурные фрагменты, потому что новость содержит имена, титулы и англоязычные вкрапления. В экономическом примере почти весь текст удерживает тема газовых поставок, а короткие переключения к бюджету и биржевым новостям возникают в местах, где речь заходит о ценах, монополиях и расчётах.


## 6. Оценка качества

Основные метрики выше уже были зафиксированы в начале ноутбука, а здесь я коротко свожу их вместе с финальными числами.

**NMI и ARI**

Эти две метрики я не использовал как целевую функцию, но оставил как сверку с редакционными рубриками `topic`.

`NMI` показывает, насколько похожи два разбиения корпуса: найденные темы и исходные рубрики `Lenta.ru`. Удобно думать о ней так:

$$
NMI = \frac{\text{общая информация между двумя разбиениями}}{\text{средний объём информации в этих разбиениях}}
$$

Значение `0` означает почти полное отсутствие связи, а значение `1` — почти полное совпадение.

`ARI` тоже сравнивает два разбиения, но делает это через пары документов:

$$
ARI = \frac{\text{насколько часто два разбиения одинаково группируют пары документов} - \text{случайное совпадение}}{\text{максимально возможное совпадение} - \text{случайное совпадение}}
$$

Значение около `0` соответствует примерно случайному совпадению, а значение `1` — почти полному совпадению.

Высокие значения `NMI` и `ARI` были бы ожидаемы, если бы модель почти один в один восстановила рубрики `Lenta.ru`. В этой работе модель строит более тонкое тематическое разбиение, поэтому эти метрики полезно трактовать как внешнюю справку, а не как основной критерий качества.



In [24]:
final_quality_table = pd.DataFrame(
    [
        {
            "topic_count": final_metrics["topic_count"],
            "coverage": final_metrics["coverage"],
            "topic_diversity": final_metrics["topic_diversity"],
            "coherence_umass": final_metrics["coherence_umass"],
            "nmi_topic": final_metrics["nmi_topic"],
            "ari_topic": final_metrics["ari_topic"],
        }
    ]
).round(4)

confirmation_view = confirmation_df[
    [
        "experiment_id",
        "topic_count",
        "coverage",
        "topic_diversity",
        "coherence_umass",
        "mean_nmi_topic",
        "mean_ari_topic",
        "selection_score",
    ]
].round(4)

display(final_quality_table)
display(confirmation_view)



,topic_count,coverage,topic_diversity,coherence_umass,nmi_topic,ari_topic
0,40,1.0,0.9975,-2.4023,0.3666,0.1444


,experiment_id,topic_count,coverage,topic_diversity,coherence_umass,mean_nmi_topic,mean_ari_topic,selection_score
0,tuned_peri_struct_text_only_light_text_only_li...,40.0,1.0,0.9933,-2.5168,0.3716,0.1398,0.7000
1,tuned_peri_struct_text_only_light_title_text_l...,40.0,1.0,0.9942,-2.5202,0.3718,0.1452,0.7476
2,tuned_peri_struct_text_only_light_title_text_l...,40.0,1.0,0.9975,-2.6501,0.3816,0.1628,0.5000


У финальной модели сильные основные метрики: полное покрытие корпуса, высокий `Topic Diversity` и `UMass Coherence = -2.4023`, который оказался заметно лучше, чем у ранних конфигураций. Это означает, что темы и различаются между собой, и опираются на устойчивые совместные появления слов в корпусе.

Дополнительные метрики с рубриками Lenta.ru дают `NMI = 0.3666` и `ARI = 0.1444`. После просмотра карты и самих тем эти значения выглядят понятными. Компактные острова спорта, науки, авиации и культуры отделяются очень чисто, а центральный политико-криминальный блок устроен плотнее и даёт больше пересечений с исходными рубриками. Для BERTopic на новостном корпусе такой результат выглядит правдоподобно.


## 7. Итоговый анализ результата

Для корпуса Lenta.ru лучше всего сработал лексический вариант BERTopic: `TF-IDF + SVD` для document embeddings, `UMAP` для нелинейного сжатия, `MiniBatchKMeans` для устойчивой кластеризации и `MMR` для финального описания тем.

На экспериментах стало видно несколько важных вещей.

Во-первых, нормализация оказалась полезнее полной лемматизации на этапе embeddings. Во-вторых, текст для document embeddings и текст для описания тем стоит рассматривать раздельно: `text_only_light` и `title_text_light` в этой задаче работают лучше именно в такой комбинации.

Финальный результат с `40` темами я считаю осмысленным. По top words, representative documents и внешней сверке с рубриками видно, что модель выделяет не случайные фрагменты, а устойчивые сюжетные блоки.

Ограничения у решения тоже есть. Некоторые темы всё ещё остаются широкими, а LLM-названия еще стоит докрутить промптимгом.

